In [ ]:
import os
import numpy as np
import pandas as pd


import pickle
from pathlib import Path

In [ ]:
#working directories

data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-11-18_grouping_cohorts_by_vax_ancestry_season" 
data1 = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-11-13_get_cohort_statistic"
data2 = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-11-13_mapping_condition_cohorts_to_viral_specie"


results = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-11-18_grouping_cohorts_by_vax_ancestry_season" 
results1 = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-11-18_get_condition_vaccines"


In [ ]:
## 1. importing data

In [ ]:
#annot csv
def get_cohort_annot():
    #cohort specie and seasonal annotation
    annot_cohort = pd.read_csv(f"{data2}/viral_cohort_person_id_overlap_annotated.csv")

    return annot_cohort

In [ ]:
def get_data_pkl(path, filename):

    # …later, in any notebook in the same workspace…
    out_file = Path(f'{path}/{filename}')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict

In [ ]:
## 2. wrangling data

In [ ]:
def get_cond_cohort_df_dict(all_cond_df): 
    print("Grouping DataFrame into a dictionary...")

    # Define the columns you want to use for your composite key
    key_columns = ['condition_concept_id', 'standard_concept_name']

    # Use a dictionary comprehension with groupby to create the dictionary
    # - The 'key' will be a tuple: (condition_concept_id, standard_concept_name)
    # - The 'group_df' will be the DataFrame containing all rows for that key
    concept_groups_dict = {
        key: group_df 
        for key, group_df in all_cond_df.groupby(key_columns)
    }

    print(f"Successfully created a dictionary with {len(concept_groups_dict)} unique (ID, Name) keys.")
    
    return concept_groups_dict

In [ ]:
def filter_cond_df_dict(cohort_dict):
    
    
    master_df_of_final_cohorts = annot_cohort.copy()

    master_key_cols = ['condition_concept_id', 'standard_concept_name']

    # Create a set of tuples (key1, key2) from the master DataFrame.
    # This set will act as our "allow list".
    valid_keys_set = set(
        master_df_of_final_cohorts[master_key_cols].itertuples(index=False, name=None)
    )
    
    # 'concept_groups_dict' is the dictionary you created in the previous step
    # 'valid_keys_set' is the set we just created

    filtered_concept_dict = {
        key: group_df 
        for key, group_df in cohort_dict.items() 
        if key in valid_keys_set
    }

    print(f"Original dictionary had {len(cohort_dict)} items.")
    print(f"Filtered dictionary now has {len(filtered_concept_dict)} items.")

    # You can now work with your new, smaller dictionary
    # print(list(filtered_concept_dict.keys())[:5])
    
    return filtered_concept_dict

In [ ]:
def wrangle_cohort_df_dict(filtered_cohort_dict): 

    cohort_dict = {}
        
    for key, df in filtered_cohort_dict.items():

        # 2. Sort by person_id and date so 'first' and 'last' work correctly
        df = df.sort_values(by=['person_id', 'condition_start_datetime'])

        # 3. Group by person_id and aggregate the data
        #    - 'first' gets the earliest diagnosis
        #    - 'last' gets the latest diagnosis
        #    - 'nunique' counts the number of unique visit IDs
        #    - 'size' counts the total number of rows (unique inputs)
        result = df.groupby('person_id').agg(
            condition_concept_id =('condition_concept_id', "first"),
            standard_concept_name=('standard_concept_name', "first"),
            first_diagnosis_date=('condition_start_datetime', 'first'),
            last_diagnosis_date=('condition_start_datetime', 'last'),
            number_of_visit_occurences=('visit_occurrence_id', 'nunique'),  # Counts unique visit IDs
            total_diagnosis_of_concept_id=('condition_start_datetime', 'nunique')                    # Counts total rows for the person
        ).reset_index()

        
        # 3. Logic: Blank out Last Date if it equals First Date
        result['last_diagnosis_date'] = np.where(
            result['first_diagnosis_date'] == result['last_diagnosis_date'], 
            pd.NaT, 
            result['last_diagnosis_date']
        )
       
    # Display the result
        cohort_dict[key] = result
    
    return cohort_dict

In [ ]:
def merge_data_table_w_cond_dict(wrangled_cohort_dict): 
    
    cohort_dict = {}

    
    # call demo and socio function for table and drop duplicates by person_id: one-row-per-patient tables once

   
    # Start with your aggregated result dataframe

    # Loop through the dictionary items
    for key, sub_df in wrangled_cohort_dict.items():
        # Merge each dataframe on 'person_id'
        final_df = sub_df.merge(demo, on='person_id', how='left')
    

        cohort_dict[key] = final_df
       

    return cohort_dict

In [ ]:
def get_cohort_updated_race(df):
    
    # Make a copy to avoid modifying the original DataFrame
    wrangled_df = df.copy()

    # --- 1. Merge Race/Ethnicity ---
    
    # Define the helper function for merging
    def merge_race_ethnicity_data(row):
        """Helper function to apply row-wise."""
        if row["ethnicity"] == "Hispanic or Latino":
            return row["ethnicity"]
        else:
            return row["race"]

    # Apply the function to create the 'updated_race' column
    wrangled_df["updated_race"] = wrangled_df.apply(merge_race_ethnicity_data, axis=1)

    # --- 2. Filter Excluded Groups ---
    
    # List of races to exclude
    to_drop = [
        'American Indian or Alaska Native',
    ]

    # Keep only rows whose updated_race is NOT in the to_drop list
    wrangled_df = wrangled_df[~wrangled_df['updated_race'].isin(to_drop)]


    return wrangled_df

In [ ]:
def get_updated_race_stats():
    
    wrangled_dict = dict()
    for name, df in ns_dict.items():
        updated = get_cohort_updated_race(df)

        wrangled_dict[name] = updated
    return wrangled_dict

In [ ]:
## 3. Bin nonseasonal cohorts

In [ ]:
def filter_dict_by_ns_concept_id(df_dict):
    
    df1 = annot_cohort.copy()
    
    nonseasonal_table =  df1.loc[:, ['condition_concept_id','standard_concept_name','non_seasonal'] ]
    ns = nonseasonal_table.loc[nonseasonal_table['non_seasonal'] == 'Y', :]
    
    # Uses the 'concept_id' column in key_df and in each df
    allowed = pd.to_numeric(ns["condition_concept_id"], errors="coerce")
    allowed = set(allowed.astype("int64"))

    out = {}
    for name, df in df_dict.items():
        mask = pd.to_numeric(df["condition_concept_id"], errors="coerce").isin(allowed)
        out[name] = df.loc[mask].copy()
    return out


In [ ]:
def get_ns_ancestry_count(nonseasonal_df):
    
    ns_cohort_bin_dict = {}
    
    for key, table in nonseasonal_df.items():
        
        
        race_count = table["updated_race"].value_counts()
        df1 = pd.DataFrame(race_count)
        df1['condition_concept_id'] = key[0] #make a concept_id column with key values
        df1['standard_concept_name'] = key[1]
        df2 = df1.reset_index() #make race a column
        
        ns_cohort_bin_dict[key] = df2
        
        
        
    ns_table = pd.concat(ns_cohort_bin_dict.values(), axis = 0, ignore_index = True)
    new_order = ['condition_concept_id', 'standard_concept_name','updated_race', 'count']
    ns_table = ns_table[new_order]
    
        # 1) List of races to exclude
    to_drop = [
        'PMI: Skip',
        'None of these',
        'American Indian or Alaska Native',
        'I prefer not to answer',
        "More than one population" 
    ]

    # 2) Keep only rows whose updated_race is not in that list
    df_filtered = ns_table[~ns_table['updated_race'].isin(to_drop)]


    
    
    
        
    return df_filtered

In [ ]:
#takes each concept and assign it to a group based on the # of ancestry groups => 100 
def get_ns_concept_ancestry_bins():
    
    # assume your DataFrame is called df and looks like:
    #   concept_id  concept_name    updated_race  count
    # 0      440029  Viral disease   White         18913
    # …      …       …               …             …

    # 1) Flag which race‐rows meet the ≥100 threshold
    df_filtered = ns_anc_count.copy()

    df_filtered['race_ge_100'] = df_filtered['count'] >= 100

    # 2) Count, per concept, how many races are ≥100
    concept_counts = (
        df_filtered 
          .groupby(['condition_concept_id','standard_concept_name'], as_index=False)
          .agg(n_eligible_races=('race_ge_100','sum'))
    )

    # 3) Bin each concept:
    #    B1: ≥2 races ≥100
    #    B2: exactly 1 race ≥100
    #    A1:  0 races ≥100
    conds = [
        concept_counts['n_eligible_races'] >= 2,
        concept_counts['n_eligible_races'] == 1,
        concept_counts['n_eligible_races'] == 0,
    ]
    choices = ['B1','B2','A1']
    concept_counts['bin'] = np.select(conds, choices, default='A1')

    
    # 4) (Optional) Merge the bin back onto your original rows
    df_binned = df_filtered.merge(
        concept_counts[['standard_concept_name','bin']],
        on='standard_concept_name',
        how='left'
    )
    
    '''
    # 5) Inspect
    print(concept_counts['bin'].value_counts())


    for grp in ['B1','B2','A1']:
        members = concept_counts.loc[concept_counts['bin']==grp, ['condition_concept_id','standard_concept_name']]
        print(f"\n=== {grp} ({len(members)} concepts) ===")
        print(members)
    '''
    
    return df_binned


In [ ]:
#getting person_Id from filtering

def get_ns_ancestry_bin_person_ids(): 
    master = pd.concat(
        [
            df.assign(
                concept_id=cid_name[0],
                concept_name=cid_name[1]
            )
            for cid_name, df in ns_cohorts.items()
        ],
        ignore_index=True
    )


    # 1) Filter your binned summary to only keep races with ≥100 people
    df_binned_filtered = ns_bin[ns_bin['race_ge_100']]

    
    
    # 2) Merge in person_id (only for those high-N rows)
    joined = df_binned_filtered.merge(
        master[
          ['condition_concept_id',
           'standard_concept_name',
           'updated_race',
           'person_id']
        ],
        on=['condition_concept_id',
            'standard_concept_name',
            'updated_race'],
        how='left'
    )

    # 3) Aggregate each (concept, race, bin) into a list of person_ids
    person_lists = (
        joined
        .groupby(
            ['condition_concept_id',
             'standard_concept_name',
             'updated_race',
             'bin'],
            as_index=False
        )['person_id']
        .apply(list)
        .rename(columns={'person_id':'person_ids'})
    )

    # 4) Split into a dict of DataFrames by bin
    dfs_by_bin = {
        b: df_bin.reset_index(drop=True)
        for b, df_bin in person_lists.groupby('bin')
    }

    
    # ─── Usage ─────────────────────────────────────────────────────────────────────
    # DataFrame for bin "A1":
    df_B1 = dfs_by_bin['B1']
    df_B2 = dfs_by_bin['B2']
    


    return df_binned_filtered, df_B1, df_B2


In [ ]:
# ─── Inputs ────────────────────────────────────────────────────────────────────
# df_binned_filtered: your summary DF already filtered to race_ge_100 == True,
#   with columns ['condition_concept_id','standard_concept_name','updated_race',…]
# ns_df: dict mapping (condition_concept_id, standard_concept_name) → the full person-level DF
#
# ─── Build your triple-keyed dict ──────────────────────────────────────────────

def get_bin_df_for_each_ancestry_and_concept(): 

    filtered_by_triple = {}
    for row in ns_bin_filtered.itertuples(index=False):
        cid, name, race = (
            row.condition_concept_id,
            row.standard_concept_name,
            row.updated_race
        )
        # grab the master DF for that concept
        df_persons = ns_cohorts[(cid, name)]
        # filter it down to just that race
        df_race = df_persons[df_persons['updated_race'] == race].copy()
        # store under the triple key
        filtered_by_triple[(cid, name, race)] = df_race
        
    return filtered_by_triple


In [ ]:
## 5. Bin nonseasonal - Vaccinated cohorts

In [ ]:
def get_ns_ethnicity_vax_binning(ns_vax_df):
    
    ns_cohort_bin_dict = {}
    
    for key, table in ns_vax_df.items():
        
        
        grouped = table.groupby('updated_race')[['vaccinated']].value_counts()
     

        df1 = pd.DataFrame(grouped)
        df1['concept_id'] = key[0] #make a concept_id column with key values
        df1['concept_name'] = key[1]
        df2 = df1.reset_index()
        
        ns_cohort_bin_dict[key] = df2
        
        
        
    ns_table = pd.concat(ns_cohort_bin_dict.values(), axis = 0, ignore_index = True)
    new_order = ['concept_id', 'concept_name','updated_race', 'vaccinated', 'count']
    ns_table = ns_table[new_order]
    final = ns_table.rename(columns={'concept_id': 'condition_concept_id_x', 'concept_name': 'standard_concept_name_x'})
    
    # 1) List of races to exclude
    to_drop = [
        'PMI: Skip',
        'None of these',
        'American Indian or Alaska Native',
        'I prefer not to answer',
        "More than one population" 
    ]

    # 2) Keep only rows whose updated_race is not in that list
    df_filtered = final [~final ['updated_race'].isin(to_drop)]    
        
        
        
    return df_filtered


In [ ]:

def get_concepts_per_bin():
    # -----------------------------------------------------------------------------
    # 1) Build per-(concept, race, vax) counts (no pre-filtering)
    # -----------------------------------------------------------------------------
    race_vax = (
        ns_ethnicity_vax_bin_count
          .groupby(
              ['standard_concept_name_x', 'updated_race', 'vaccinated'],
              as_index=False
          )['count']
          .sum()
          .pivot_table(
              index=['standard_concept_name_x', 'updated_race'],
              columns='vaccinated',
              values='count',
              fill_value=0
          )
          .reset_index()
          .rename(columns={'N': 'N', 'Y': 'Y'})
    )

    # compute total patients per race
    race_vax['race_total'] = race_vax['Y'] + race_vax['N']

    # -----------------------------------------------------------------------------
    # 2) Collapse to one row per concept with summary features
    # -----------------------------------------------------------------------------
    concept_summary = (
        race_vax
          .groupby('standard_concept_name_x', as_index=False)
          .agg(
            total_races     = ('updated_race', 'nunique'),                  # number of distinct races
            complete_races  = ('race_total',  lambda x: ((race_vax.loc[x.index,'Y'] > 100) &
                                                         (race_vax.loc[x.index,'N'] > 100)).sum()),
            eligible_races  = ('race_total',  lambda x: (x >= 100).sum())    # races with total ≥100
          )
    )

    # -----------------------------------------------------------------------------
    # 3) Define cohorts C1–D3
    # -----------------------------------------------------------------------------
    conds = [
        # C1: ≥2 races each with Y>100 & N>100
        concept_summary['complete_races'] >= 2,

        # C2: exactly 1 race with Y>100 & N>100
        concept_summary['complete_races'] == 1,

        # D1: no complete races, but ≥2 races with total ≥100
        (concept_summary['complete_races'] == 0) &
        (concept_summary['eligible_races']  >= 2),

        # D2: no complete races, exactly 1 race with total ≥100
        (concept_summary['complete_races'] == 0) &
        (concept_summary['eligible_races']  == 1),

    ]
    choices = ['C1', 'C2', 'D1', 'D2']

    # anything else → D3
    concept_summary['cohort_group'] = np.select(conds, choices, default='D3')

    # -----------------------------------------------------------------------------
    # 4) (Optional) Merge cohort_group back onto the raw DataFrame
    # -----------------------------------------------------------------------------
    df_all = (
        ns_ethnicity_vax_bin_count
          .merge(
              concept_summary[['standard_concept_name_x', 'cohort_group']],
              on='standard_concept_name_x',
              how='left'
          )
    )

    # -----------------------------------------------------------------------------
    # 5) Sanity checks
    # -----------------------------------------------------------------------------
    print("Concepts per cohort:")
    print(concept_summary['cohort_group'].value_counts(), "\n")

    print("Examples by cohort:")
    for grp in ['C1','C2','D1','D2','D3']:
        ex = concept_summary.loc[
            concept_summary['cohort_group'] == grp, 'standard_concept_name_x'
        ].unique()[:]
        print(f" {grp}: {ex}")
        
        
    return  df_all, concept_summary

In [ ]:
##getting person_Id from filtering

In [ ]:
#C1 and C2
# ─── INPUTS ────────────────────────────────────────────────────────────────────
    # 1) ns_vax_df: dict mapping
    #      (condition_concept_id, standard_concept_name)
    #    → full person-level DataFrame with at least:
    #      ['person_id', 'updated_race', 'vaccinated', …]
    #
    # 2) df_all: DataFrame with columns
    #      ['condition_concept_id_x',
    #       'standard_concept_name_x',
    #       'updated_race',
    #       'vaccinated',
    #       'count',
    #       'cohort_group']
    #    as produced in your step (4). We’ll re-derive race-level completeness from its counts.

# ─── 1) Compute per-(concept, race) N/Y counts & mark “complete” races ─────────

def get_c1_c2_cohort_dfs():
    
        race_counts = (
            df_all
            .pivot_table(
                index=['condition_concept_id_x', 'standard_concept_name_x', 'updated_race'],
                columns='vaccinated',
                values='count',
                aggfunc='sum',
                fill_value=0
            )
            .reset_index()
            .rename(columns={'N': 'N_count', 'Y': 'Y_count'})
        )
        race_counts['complete_race'] = (
            (race_counts['N_count'] > 100) &
            (race_counts['Y_count'] > 100)
        )

        # ─── 2) Summarize per-concept how many “complete” races there are & assign C1/C2 ─
        concept_summary = (
            race_counts
            .groupby(
                ['condition_concept_id_x', 'standard_concept_name_x'],
                as_index=False
            )
            .agg(complete_races=('complete_race', 'sum'))
        )
        conds = [
            concept_summary['complete_races'] >= 2,   # C1
            concept_summary['complete_races'] == 1,   # C2
        ]
        choices = ['C1', 'C2']
        concept_summary['cohort_group'] = np.select(conds, choices, default=None)

        # keep only the C1/C2 concepts
        concept_summary = concept_summary[
            concept_summary['cohort_group'].isin(['C1','C2'])
        ].copy()

        # ─── 3) Filter race_counts to only those complete races of C1/C2 concepts ──────
        race_complete = (
            race_counts
            .merge(
                concept_summary,
                on=['condition_concept_id_x', 'standard_concept_name_x'],
                how='inner'
            )
        )
        race_complete = race_complete[race_complete['complete_race']].copy()

        # ─── 4) Build your dict of person-level DataFrames for each slice ──────────────
        # Key = (concept_id, concept_name, race, vaccinated_flag, cohort_group)
        vax_person_dfs = {}
        for row in race_complete.itertuples(index=False):
            cid   = row.condition_concept_id_x
            name  = row.standard_concept_name_x
            race  = row.updated_race
            cohort = row.cohort_group

            # grab full person-level DF for this concept
            df_full = ns_vax_df[(cid, name)]

            # split out N and Y for this race
            for flag in ['N','Y']:
                df_slice = df_full[
                    (df_full['updated_race'] == race) &
                    (df_full['vaccinated']     == flag)
                ].copy()

                key = (cid, name, race, flag, cohort)
                vax_person_dfs[key] = df_slice


        return vax_person_dfs

In [ ]:
##D1, D2



## ─── INPUTS ────────────────────────────────────────────────────────────────────
# 1) ns_vax_df: dict mapping
#      (condition_concept_id, standard_concept_name)
#    → full person-level DataFrame, with columns at least:
#      ['person_id', 'updated_race', 'vaccinated', …]
#
# 2) df_all: DataFrame with columns
#      ['condition_concept_id_x',
#       'standard_concept_name_x',
#       'updated_race',
#       'vaccinated',
#       'count']
#    (i.e. the per-race, per-vax counts you computed earlier)

# ─── 1) Compute per-(concept, race) totals & mark “complete” races ────────────

def get_d1_d2_cohort_dfs():
    race_counts = (
        df_all
          .pivot_table(
              index=['condition_concept_id_x', 'standard_concept_name_x', 'updated_race'],
              columns='vaccinated',
              values='count',
              aggfunc='sum',
              fill_value=0
          )
          .reset_index()
    )
    race_counts.columns.name = None
    race_counts = race_counts.rename(columns={'N': 'N_count', 'Y': 'Y_count'})
    race_counts['race_total']    = race_counts['N_count'] + race_counts['Y_count']
    race_counts['complete_race'] = (
        (race_counts['N_count'] > 100) &
        (race_counts['Y_count'] > 100)
    )

    # ─── 2) Summarize per-concept → how many complete vs. eligible races ──────────
    concept_summary = (
        race_counts
          .groupby(
              ['condition_concept_id_x', 'standard_concept_name_x'],
              as_index=False
          )
          .agg(
            complete_races = ('complete_race', 'sum'),
            eligible_races = ('race_total', lambda x: (x >= 100).sum())
          )
    )

    # assign D1 / D2
    conds = [
        (concept_summary['complete_races'] == 0) &
          (concept_summary['eligible_races'] >= 2),  # D1
        (concept_summary['complete_races'] == 0) &
          (concept_summary['eligible_races'] == 1)   # D2
    ]
    choices = ['D1','D2']
    concept_summary['cohort_group'] = np.select(conds, choices, default=None)

    # keep only D1/D2 concepts
    d_concepts = concept_summary[
        concept_summary['cohort_group'].isin(['D1','D2'])
    ].copy()

    # ─── 3) For those concepts, pick only the eligible races (race_total ≥100) ────
    d_races = (
        race_counts
          .merge(
              d_concepts,
              on=['condition_concept_id_x','standard_concept_name_x'],
              how='inner'
          )
    )
    d_races = d_races[d_races['race_total'] >= 100].copy()

    # ─── 4) Build dict of person-level DFs for each (concept, race, cohort) ───────
    # Key = (condition_concept_id, concept_name, updated_race, cohort_group)
    person_dfs = {}
    for row in d_races.itertuples(index=False):
        cid    = row.condition_concept_id_x
        name   = row.standard_concept_name_x
        race   = row.updated_race
        cohort = row.cohort_group

        # original full DF for this concept
        df_full = ns_vax_df[(cid, name)]

        # keep ONLY that race (both vaccinated & unvaccinated)
        df_slice = df_full[df_full['updated_race'] == race].copy()

        person_dfs[(cid, name, race, cohort)] = df_slice

    return person_dfs

In [ ]:
#raw Vaccinated Y / N cocnepts

In [ ]:
def get_ns_vax_binning(ns_vax_df):
    
    ns_cohort_bin_dict = {}
    
    for key, table in ns_vax_df.items():
        
        
        grouped = table.groupby('updated_race')[['vaccinated']].value_counts()
     

        df1 = pd.DataFrame(grouped)
        df1['concept_id'] = key[0] #make a concept_id column with key values
        df1['concept_name'] = key[1]
        df2 = df1.reset_index()
        
        ns_cohort_bin_dict[key] = df2
        
        
        
    ns_table = pd.concat(ns_cohort_bin_dict.values(), axis = 0, ignore_index = True)
    new_order = ['concept_id', 'concept_name','updated_race', 'vaccinated', 'count']
    ns_table = ns_table[new_order]
    final = ns_table.rename(columns={'concept_id': 'condition_concept_id_x', 'concept_name': 'standard_concept_name_x'})
    
    # 1) List of races to exclude
    to_drop = [
        'PMI: Skip',
        'None of these',
        'American Indian or Alaska Native',
        'I prefer not to answer',
        "More than one population" 
    ]

    # 2) Keep only rows whose updated_race is not in that list
    df_filtered = final [~final ['updated_race'].isin(to_drop)]    
        
        
        
    return df_filtered


In [ ]:
def get_raw_vax_y_n_bins():
    vax_df = (
        ns_vax_bin_count
          .groupby(
              ['standard_concept_name_x', 'vaccinated'],
              as_index=False
          )['count']
          .sum()
          .pivot_table(
              index=['standard_concept_name_x'],
              columns='vaccinated',
              values='count',
              fill_value=0
          )
          .reset_index()
          .rename(columns={'N': 'N', 'Y': 'Y'})
    )

    # compute total patients per race
    vax_df['vax_total'] = vax_df['Y'] + vax_df['N']

    #removing rows where Y and N arent >= 100
    vax_cohort = vax_df.loc[(vax_df['Y'] >= 100) & (vax_df['N'] >= 100), :]
    
    return vax_cohort     
    
    

In [ ]:

# ─── 0) INPUTS ─────────────────────────────────────────────────────────────────
# ns_vax_df: dict mapping (concept_id, concept_name) → person-level DataFrame
#   each DF must have at least these columns:
#     ['person_id', 'vaccinated', …]
#
# vax_cohort: DataFrame with columns
#   ['condition_concept_id', 'standard_concept_name', 'N', 'Y', 'vax_total']
#   listing only the concepts where both N ≥ 100 and Y ≥ 100.
#
# If your vax_cohort only has 'standard_concept_name' (no concept_id),
# you should re-compute it including the id:

# Flatten and recompute if needed:

def get_raw_bin_cohort_dfs():
    master_vax = pd.concat(
        [
            df.assign(
                condition_concept_id=key[0],
                standard_concept_name=key[1]
            )
            for key, df in ns_vax_df.items()
        ],
        ignore_index=True
    )

    # Recompute counts by both id+name+flag:
    vax_counts = (
        master_vax
        .groupby(
            ['condition_concept_id', 'standard_concept_name', 'vaccinated'],
            as_index=False
        )['person_id']
        .count()
        .rename(columns={'person_id': 'count'})
        .pivot_table(
            index=['condition_concept_id', 'standard_concept_name'],
            columns='vaccinated',
            values='count',
            fill_value=0
        )
        .reset_index()
        .rename(columns={'N': 'N_count', 'Y': 'Y_count'})
    )
    vax_counts['vax_total'] = vax_counts['N_count'] + vax_counts['Y_count']

    # Filter to your ≥100 threshold:
    vax_cohort = vax_counts.loc[
        (vax_counts['N_count'] >= 100) &
        (vax_counts['Y_count'] >= 100),
        ['condition_concept_id', 'standard_concept_name']
    ]

    # ─── 1) Build the triple-key dict ───────────────────────────────────────────────
    # Keys = (condition_concept_id, standard_concept_name, vaccinated_flag)
    # Values = DataFrame of all person-level rows for that slice
    cohort_set = set(
        zip(
            vax_cohort['condition_concept_id'],
            vax_cohort['standard_concept_name']
        )
    )

    vax_person_dfs1 = {}
    for (cid, name) in cohort_set:
        df_full = ns_vax_df[(cid, name)]
        for flag in ['N', 'Y']:
            df_slice = df_full[df_full['vaccinated'] == flag].copy()
            # keep whatever columns you need—e.g. person_id, plus any others
            vax_person_dfs1[(cid, name, flag)] = df_slice

    # ─── USAGE ─────────────────────────────────────────────────────────────────────
    # What keys do we have?
    print("Available slices:", list(vax_person_dfs1.keys()))

    print(len(list(vax_person_dfs1.keys())))
    
    return vax_person_dfs1

In [ ]:
## 6. Bin Covid-19 cohorts

In [ ]:
def get_covid_vax_bins_by_strain_date():
    
    
        covid_vax_df = ns_vax_df[(37311061, 'COVID-19')]

        # define your fixed calendar bins as date pairs
        bins = {
            'Wuhan':     (pd.to_datetime('2020-01-01').date(),
                                 pd.to_datetime('2020-12-31').date()),
            'Alpha':     (pd.to_datetime('2021-01-01').date(),
                                 pd.to_datetime('2021-06-30').date()),
            'Delta':     (pd.to_datetime('2021-07-01').date(),
                                 pd.to_datetime('2021-12-31').date()),
            'Omicron': (pd.to_datetime('2021-12-01').date(),
                                 pd.to_datetime('2023-10-31').date()),
        }

        all_counts = []

        df = covid_vax_df.copy()

        # convert to date only (drops time & tz)
        df['diag_date'] = pd.to_datetime(df['first_dx']).dt.date

        # assign each row to exactly one bin (or None)
        def assign_bin(d):
            for label, (start, end) in bins.items():
                if start <= d <= end:
                    return label
            return None

        df['bin'] = df['diag_date'].apply(assign_bin)
        df = df.dropna(subset=['bin'])

        # count one row per person by race & vax status in each bin
        counts = (
            df.groupby(['bin', 'updated_race', 'vaccinated'])
              .size()
              .reset_index(name='count')
        )
        counts['condition_concept_id']   = df["condition_concept_id"]
        counts['standard_concept_name'] = df['standard_concept_name_baseline']

        all_counts.append(counts)

        # stitch all results together
        result = pd.concat(all_counts, ignore_index=True)
    
        results = result.loc[result['count'] >= 100, :]
    
    
        return results




In [ ]:
def get_covid_y_n_vax_grps():
    
    def has_Y_and_N(group):
        vax_set = set(group['vaccinated'])
        return vax_set == {'Y', 'N'} and len(group) == 2

    # Apply groupby and filter
    filtered_df = covid_df_bin.groupby(['bin','updated_race', 'condition_concept_id']).filter(has_Y_and_N)

    df1 = pd.DataFrame(filtered_df)
    return df1

In [ ]:
def build_covid_strain_person_dict():
        # 1) define the fixed calendar bins
        bins = {
            'Wuhan':   (pd.to_datetime('2020-01-01').date(),
                        pd.to_datetime('2020-12-31').date()),
            'Alpha':   (pd.to_datetime('2021-01-01').date(),
                        pd.to_datetime('2021-06-30').date()),
            'Delta':   (pd.to_datetime('2021-07-01').date(),
                        pd.to_datetime('2021-12-31').date()),
            'Omicron': (pd.to_datetime('2021-12-01').date(),
                        pd.to_datetime('2023-10-31').date()),
        }

        # 2) flatten and assign each row to a strain‐bin
        records = []
        
        df = ns_vax_df[(37311061, 'COVID-19')]
        tmp = df.copy()
        
        
        
        
        
        
        
        
        tmp['diag_date'] = pd.to_datetime(tmp['first_dx']).dt.date

        def assign_bin(d):
            for label, (start, end) in bins.items():
                if start <= d <= end:
                    return label
            return None

        tmp['bin'] = tmp['diag_date'].apply(assign_bin)
        tmp = tmp.dropna(subset=['bin']).copy()
        tmp['concept_id']   = tmp["condition_concept_id"]
        tmp['concept_name'] = tmp['standard_concept_name_baseline']
        records.append(tmp)

        master = pd.concat(records, ignore_index=True)

        # 3) count per (concept, bin, race, vaccinated)
        counts = (
            master
            .groupby(
                ['concept_id', 'concept_name', 'bin', 'updated_race', 'vaccinated'],
                as_index=False
            )['person_id']
            .count()
            .rename(columns={'person_id': 'count'})
        )

        # 4) filter to count ≥ 100
        counts = counts[counts['count'] >= 100].copy()

        # 5) keep only slices having both Y and N
        def has_both(group):
            return set(group['vaccinated']) == {'Y','N'}
        valid = (
            counts
            .groupby(['concept_id','concept_name','bin','updated_race'])
            .filter(has_both)
        )

        # 6) build dict including vaccinated in the key
        covid_person_dfs = {}
        for row in valid.itertuples(index=False):
            key = (
                row.concept_id,
                row.concept_name,
                row.bin,
                row.updated_race,
                row.vaccinated
            )
            df_slice = master[
                (master['concept_id']   == row.concept_id) &
                (master['concept_name'] == row.concept_name) &
                (master['bin']          == row.bin) &
                (master['updated_race'] == row.updated_race) &
                (master['vaccinated']   == row.vaccinated)
            ].copy()
            covid_person_dfs[key] = df_slice

        return covid_person_dfs


In [ ]:
## 7. Bin Influenza cohort

In [ ]:
import pandas as pd

def get_flu_vax_seasonal_binning():
    seasonal_vax_cohort_bin_dict = {}
    
        # define your epi‐season bounds
    start_week = 40   # week 40 of the season year
    end_week   = 20   # week 20 of the following year

    table = ns_vax_df[(46273463, 'Upper respiratory tract infection due to Influenza')].copy()
    
        
        # extract ISO week and year
    iso = table['first_dx'].dt.isocalendar()
    week = iso.week
    year = iso.year

        # assign each row to the season_year in which its season started:
        #  - if week >= start_week → belongs to season starting that calendar year
        #  - else                → belongs to season that started the previous calendar year
    table['season_year'] = year.where(week >= start_week, year - 1)

        # keep only rows in the seasonal window (week >= 40 OR week <= 20)
    in_window = (week >= start_week) | (week <= end_week)
    df_window = table.loc[in_window].copy()

        # count updated_race by season_year
    seasonal_vax_count = (
            df_window
              .groupby('season_year')[['updated_race','vaccinated']]
              .value_counts()
        )

        # turn into a DataFrame and attach concept info
    df_counts = seasonal_vax_count.rename('count').reset_index()
    df_counts['concept_id']   = table["condition_concept_id"]
    df_counts['concept_name'] = table["standard_concept_name_baseline"]

    seasonal_vax_cohort_bin_dict[(46273463, 'Upper respiratory tract infection due to Influenza')] = df_counts

    # concatenate all concepts’ results into one table
    seasonal_vax_table = pd.concat(
        seasonal_vax_cohort_bin_dict.values(),
        axis=0,
        ignore_index=True
    )
    
    
    to_drop = [
        'PMI: Skip',
        'None of these',
        'American Indian or Alaska Native',
        'I prefer not to answer',
        "More than one population" 
    ]

    # 2) Keep only rows whose updated_race is not in that list
    flu_race_filtered = seasonal_vax_table[~seasonal_vax_table['updated_race'].isin(to_drop)]   
    
    final_flu_bins = flu_race_filtered.loc[flu_race_filtered['count'] >= 100, :]
    
    def has_Y_and_N(group):
            vax_set = set(group['vaccinated'])
            return vax_set == {'Y', 'N'} and len(group) == 2

            # Apply groupby and filter
    filtered_df = final_flu_bins.groupby(['season_year','updated_race']).filter(has_Y_and_N)

    df1 = pd.DataFrame(filtered_df)


#    return df1
    return  df1
    



In [ ]:
import pandas as pd

def get_flu_bin_cohorts():
  # ─── 0) INPUT: s_vax_df ────────────────────────────────────────────────────────
    # A dict mapping (concept_id, concept_name) → person-level DataFrame
    # each DF must have at least:
    #   ['person_id', 'first_diag_date_x' (datetime64), 'updated_race', 'vaccinated', …]

    # ─── 1) FLATTEN AND ASSIGN SEASON • WINDOW ─────────────────────────────────────
        records = []

        tmp = ns_vax_df[(46273463, 'Upper respiratory tract infection due to Influenza')].copy()
        iso  = tmp['first_dx'].dt.isocalendar()
        week = iso.week
        year = iso.year

        # season_year = year if week ≥ 40, else year - 1
        tmp['season_year'] = year.where(week >= 40, year - 1)

        # only keep weeks in [40..52] ∪ [1..20]
        in_window = (week >= 40) | (week <= 20)
        tmp = tmp.loc[in_window].copy()

        # tag concept
        tmp['concept_id']   = tmp["condition_concept_id"]
        tmp['concept_name'] = tmp["standard_concept_name_baseline"]

        records.append(tmp)

        master = pd.concat(records, ignore_index=True)

        # ─── 2) COUNT PER SLICE ─────────────────────────────────────────────────────────
        seasonal_counts = (
            master
            .groupby(
                ['concept_id', 'concept_name', 'season_year', 'updated_race', 'vaccinated'],
                as_index=False
            )['person_id']
            .count()
            .rename(columns={'person_id': 'count'})
        )

        # ─── 3) FILTER TO count ≥ 100 ─────────────────────────────────────────────────
        count_filter = seasonal_counts.loc[seasonal_counts['count'] >= 100].copy()

        # ─── 4) KEEP ONLY GROUPS WITH BOTH Y & N ────────────────────────────────────────
        def has_Y_and_N(g):
            s = set(g['vaccinated'])
            return s == {'Y', 'N'} and len(g) == 2

        filtered = (
            count_filter
            .groupby(['concept_id', 'concept_name', 'season_year', 'updated_race'])
            .filter(has_Y_and_N)
            .reset_index(drop=True)
        )

        # ─── 5) BUILD DICT OF PERSON-LEVEL DFs ─────────────────────────────────────────
        # Key = (concept_id, concept_name, season_year, updated_race, vaccinated)
        seasonal_person_dfs = {}
        for row in filtered.itertuples(index=False):
            key = (
                row.concept_id,
                row.concept_name,
                row.season_year,
                row.updated_race,
                row.vaccinated
            )
            sub = master[
                (master['concept_id']    == row.concept_id) &
                (master['concept_name']  == row.concept_name) &
                (master['season_year']   == row.season_year) &
                (master['updated_race']  == row.updated_race) &
                (master['vaccinated']    == row.vaccinated)
            ].copy()
            seasonal_person_dfs[key] = sub

        return seasonal_person_dfs, filtered

In [ ]:
## 8. Save data as pkls

In [ ]:
def create_df_pkl(df, directory):

    # 2) Choose a workspace folder for persistence
    out_file = Path(directory)

    # 3) Save the entire dict in one go
    with open(out_file, 'wb') as f:
        pickle.dump(df, f)

    print(f"Saved {len(df)} DataFrames to {out_file}")

In [ ]:
##Function Calls



##get data

annot_cohort = get_cohort_annot()
cohort = get_data_pkl(data1, "cohort_cond_df.pkl")
demo = get_data_pkl(data1, "cohort_demo_df.pkl")
ns_vax_df = get_data_pkl(results1, "cohort_vaccine_dict.pkl")

##wrangle data for ns cohorts

cohort_dict = get_cond_cohort_df_dict(cohort)
filtered_cohort_dict = filter_cond_df_dict(cohort_dict)
wrangled_cohort_dict = wrangle_cohort_df_dict(filtered_cohort_dict)
ns_dict = merge_data_table_w_cond_dict(wrangled_cohort_dict)
ns_dict1 = get_updated_race_stats()


##ns wrangle and binning

ns_cohorts = filter_dict_by_ns_concept_id(ns_dict1)
ns_anc_count = get_ns_ancestry_count(ns_cohorts)
ns_bin = get_ns_concept_ancestry_bins()
ns_bin_filtered, df_b1, df_b2 = get_ns_ancestry_bin_person_ids()
final_ns_df = get_bin_df_for_each_ancestry_and_concept()



ns_ethnicity_vax_bin_count = get_ns_ethnicity_vax_binning(ns_vax_df)
df_all, concept_summary = get_concepts_per_bin()
c1_c2_dict_df = get_c1_c2_cohort_dfs()
d1_d2_dict_df = get_d1_d2_cohort_dfs()


##ns vax binning

ns_vax_bin_count = get_ns_vax_binning(ns_vax_df)
raw_vax_bins =  get_raw_vax_y_n_bins()
raw_vax_cohort_dfs = get_raw_bin_cohort_dfs()



##Covid-19 binning

covid_df_bin = get_covid_vax_bins_by_strain_date()
covid_y_n_bins = get_covid_y_n_vax_grps()
covid_person_by_strain_and_vax = build_covid_strain_person_dict()


##flu binning

flu_seasonal_bin_counts = get_flu_vax_seasonal_binning()
final_flu_seasonal_bin_cohort_dict,  filtered = get_flu_bin_cohorts()


##export data pkls of binned cohort dictionary of dataframes 

create_df_pkl(ns_dict1, f'{results}/ns_cohort_dict_df.pkl')
create_df_pkl(ns_vax_df, f'{results}/ns_vax_cohort_dict_df.pkl')

create_df_pkl(final_ns_df, f'{results}/b1_b2_dict_df.pkl')

create_df_pkl(c1_c2_dict_df, f'{results}/c1_c2_dict_df.pkl')
create_df_pkl(d1_d2_dict_df, f'{results}/d1_d2_dict_df.pkl')

create_df_pkl(raw_vax_cohort_dfs, f'{results}/ns_raw_vax_cohort_dict_df.pkl')

create_df_pkl(covid_person_by_strain_and_vax, f'{results}/covid_bin_cohorts.pkl')
create_df_pkl(final_flu_seasonal_bin_cohort_dict, f'{results}/flu_bin_cohorts.pkl')




In [ ]:
##Visualization

#cohort_dict

#ns_vax_df


#filtered_cohort_dict
#wrangled_cohort_dict
#ns_dict1


#ns_anc_count
#ns_bin
#ns_bin_filtered 
#final_ns_df.keys()

#ns_ethnicity_vax_bin_count
#d1  = concept_summary.loc[concept_summary['cohort_group'] == 'D1', :].copy()
#d1
c1_c2_dict_df 
#d1_d2_dict_df

#ns_vax_bin_count
#raw_vax_bins
#raw_vax_cohort_dfs

#covid_df_bin
#covid_y_n_bins
#covid_person_by_strain_and_vax


#flu_seasonal_bin_counts


#concept_summary


In [ ]:
d1_d2_dict_df.keys()

In [ ]:
df_all